# Mol2Vec Encoder


In [ ]:
import random
import numpy as np
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


# =========================
# Reproducibility
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# =========================
# Load data
# =========================
train_df = pd.read_csv("R1D100_train.csv")
test_df  = pd.read_csv("R1D100_test.csv")

X_train = train_df.iloc[:, 1:-1].values.astype(np.float32)
X_test  = test_df.iloc[:, 1:-1].values.astype(np.float32)


# =========================
# Dataset
# =========================
class Mol2VecDataset(Dataset):
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32)


train_loader = DataLoader(
    Mol2VecDataset(X_train),
    batch_size=8,
    shuffle=True
)


class Mol2VecGRU(nn.Module):

    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()

        self.hidden_dim = hidden_dim

        # Encoder
        self.encoder = nn.GRU(
            input_size=1,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Decoder
        self.decoder = nn.GRU(
            input_size=1,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.out = nn.Linear(hidden_dim, 1)

    def encode(self, x):
        x = x.unsqueeze(-1)          
        _, h = self.encoder(x)
        return h[-1]                 

    def decode(self, z, seq_len):
        B = z.size(0)

        # repeat latent as initial context
        dec_input = torch.zeros(B, seq_len, 1, device=z.device)

        out, _ = self.decoder(dec_input, z.unsqueeze(0))
        out = self.out(out).squeeze(-1)

        return out

    def forward(self, x):
        z = self.encode(x)
        recon = self.decode(z, x.shape[1])
        return recon, z


# =========================
# Device
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Mol2VecGRU(X_train.shape[1], hidden_dim=32).to(device)


# =========================
# TRAINING
# =========================
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

EPOCHS = 100

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for x in train_loader:
        x = x.to(device)

        optimizer.zero_grad()

        recon, z = model(x)

        loss = criterion(recon, x)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")


# =========================
# EMBEDDING
# =========================
model.eval()

with torch.no_grad():

    X_train_t = torch.tensor(X_train).to(device)
    X_test_t  = torch.tensor(X_test).to(device)

    train_emb = model.encode(X_train_t).cpu().numpy()
    test_emb  = model.encode(X_test_t).cpu().numpy()


# =========================
# SAVE
# =========================
pd.DataFrame(train_emb).to_csv("mol2vec_gru_train.csv", index=False)
pd.DataFrame(test_emb).to_csv("mol2vec_gru_test.csv", index=False)

print("Done")
print(train_emb.shape, test_emb.shape)